# Memecoins: Pump or Dump — Baseline Notebook

Workflow: **Colab notebook + Google Drive + GitHub** (same as Zindi/Opencode sessions).

- Username: `kinuthiawachira` (Telegram linked ✅)
- Runtime: **CPU** (free tier)
- Data: `/content/data/`
- Persistence: `/content/drive/MyDrive/kaggle/memecoins-pump-or-dump/`

Run cells top to bottom. Do **not** copy-paste — this file lives in GitHub and Drive.

## Colab Secrets (key icon 🔑)
Add these two secrets **before** running Cell 2:
- `KAGGLE_USERNAME` — your Kaggle username
- `KAGGLE_API_TOKEN` — your Kaggle API key (kaggle.com → Account → API → Create New API Token)

Access via `from google.colab import userdata; userdata.get('secret_name')`
Enable **Notebook access** on both, then **Runtime → Restart session**.

In [ ]:
# Cell 1 — Mount Google Drive and set up project paths
from google.colab import drive
import os
from pathlib import Path

drive.mount('/content/drive', force_remount=True)

DRIVE_PROJECT = Path('/content/drive/MyDrive/kaggle/memecoins-pump-or-dump')
DATA_DIR = Path('/content/data')
OUTPUT_DIR = Path('/content/outputs')

for p in [DRIVE_PROJECT, DATA_DIR, OUTPUT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print('Drive project:', DRIVE_PROJECT)
print('Data dir:', DATA_DIR)
print('Output dir:', OUTPUT_DIR)
print('Drive contents:', [p.name for p in DRIVE_PROJECT.iterdir()])


In [ ]:
# Cell 2 — Install deps + Kaggle CLI, write kaggle.json from uploaded file
!pip install -q kaggle pandas numpy scikit-learn xgboost lightgbm

import json
from pathlib import Path

# Read kaggle.json from uploaded file in Colab files sidebar
kaggle_json_path = Path('/content/kaggle.json')

if not kaggle_json_path.exists():
    raise ValueError(
        "kaggle.json not found in Colab files sidebar.\n"
        "Upload your Kaggle API token (kaggle.com → Account → API → Create New API Token)\n"
        "to the files sidebar (folder icon 📁), then re-run this cell."
    )

with open(kaggle_json_path) as f:
    creds = json.load(f)

username = creds.get('username')
key = creds.get('key')

# Debug: presence only
print('username found:', bool(username))
print('key found:', bool(key))

if not username or not key:
    raise ValueError(
        "kaggle.json is missing 'username' or 'key' fields.\n"
        "Re-download from kaggle.com → Account → API → Create New API Token."
    )

kaggle_dir = Path('/root/.kaggle')
kaggle_dir.mkdir(parents=True, exist_ok=True)
with open(kaggle_dir / 'kaggle.json', 'w') as f:
    json.dump({"username": username, "key": key}, f)

!chmod 600 /root/.kaggle/kaggle.json
!kaggle competitions download -c memecoins-pump-or-dump -p /content/data
!unzip -o /content/data/memecoins-pump-or-dump.zip -d /content/data/

print('Files in /content/data:')
for f in sorted(Path('/content/data').iterdir()):
    print(f'  {f.name}  ({f.stat().st_size / 1024:.1f} KB)')


In [ ]:
# Cell 3 — Explore the data
import pandas as pd
import numpy as np

csv_files = list(DATA_DIR.glob('*.csv'))
print('CSV files:', [f.name for f in csv_files])

for f in csv_files:
    df = pd.read_csv(f, nrows=5)
    print(f'\n=== {f.name} ===')
    print('Shape (first 5 rows):', df.shape)
    print('Columns:', list(df.columns))
    print(df.head())


In [ ]:
# Cell 4 — Load full training data and build baseline
# TODO: update filenames after Cell 3 shows actual names
train_file = None
test_file = None
for f in csv_files:
    name = f.name.lower()
    if 'train' in name:
        train_file = f
    if 'test' in name or 'sample' in name:
        test_file = f

if train_file:
    train = pd.read_csv(train_file)
    print(f'Train shape: {train.shape}')
    print(train.dtypes)
    print(train.describe(include='all'))

if test_file:
    test = pd.read_csv(test_file)
    print(f'\nTest shape: {test.shape}')
    print(test.dtypes)
    print(test.head())


In [ ]:
# Cell 5 — Baseline: all zeros (matches sample_submission.csv)
# submission.csv = one number per token: sell target (tp), or 0 = do not buy

if test_file is not None:
    # Adjust 'token_id' to match your actual ID column name from Cell 3
    id_col = 'token_id' if 'token_id' in test.columns else test.columns[0]
    
    submission = pd.DataFrame({
        id_col: test[id_col],
        'tp': 0
    })
    
    submission_path = OUTPUT_DIR / 'submission.csv'
    submission.to_csv(submission_path, index=False)
    print(f'Saved {len(submission)} rows to {submission_path}')
    print(submission.head())
    print(f'Skip rate: {(submission["tp"] == 0).mean():.1%}')
else:
    print('No test file found yet — run Cell 3 first')


In [ ]:
# Cell 6 — Save notebook and submission to Google Drive (persists across sessions)
import shutil, datetime

timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')

# Copy submission to Drive
drive_submission = DRIVE_PROJECT / f'submission_{timestamp}.csv'
if OUTPUT_DIR.joinpath('submission.csv').exists():
    shutil.copy(OUTPUT_DIR / 'submission.csv', drive_submission)
    print(f'Saved submission to Drive: {drive_submission}')

# Save a run marker to Drive
marker = DRIVE_PROJECT / f'ran_{timestamp}.txt'
marker.write_text(f'Baseline run at {timestamp}\n')
print(f'Wrote run marker: {marker}')

print('\nDrive project contents:')
for p in sorted(DRIVE_PROJECT.iterdir()):
    print(f'  {p.name}')
